In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
import pandas as pd
import time
import json
import gzip

In [ ]:
import torch
from transformers import BertTokenizer
import pandas as pd
import json
import gzip

# Load the appliance dataset
with gzip.open('Appliances.json.gz', 'r') as f:
    df = pd.read_json(f, lines=True, nrows=20000)

# Keep only the 'overall' and 'reviewText' columns
df = df[['overall', 'reviewText']]

# Drop rows with missing values
df.dropna(subset=['reviewText'], inplace=True)

# Map the 'overall' column to sentiment labels
sentiment_to_label = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
df['sentiment'] = df['overall'].map(sentiment_to_label)

# Split the dataset into training, validation, and test sets
train_df = df.sample(frac=0.8, random_state=60)
test_df = df.drop(train_df.index)
val_df = test_df.sample(frac=0.5, random_state=60)
test_df = test_df.drop(val_df.index)

# Instantiate the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the text data and convert it to input features for BERT
train_encodings = tokenizer(list(train_df['reviewText'].tolist()), truncation=True, padding=True, return_tensors='pt')
val_encodings = tokenizer(list(val_df['reviewText'].tolist()), truncation=True, padding=True, return_tensors='pt')
test_encodings = tokenizer(list(test_df['reviewText'].tolist()), truncation=True, padding=True, return_tensors='pt')

# Convert the sentiment labels to numerical values (0 to 4)
train_labels = torch.tensor(list(train_df['sentiment']))
val_labels = torch.tensor(list(val_df['sentiment']))
test_labels = torch.tensor(list(test_df['sentiment']))

# Create PyTorch datasets from the input features and labels
train_dataset = torch.utils.data.TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels)
val_dataset = torch.utils.data.TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'], val_labels)
test_dataset = torch.utils.data.TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], test_labels)



In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader
torch.cuda.set_per_process_memory_fraction(0.8, 0)
# Instantiate the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Instantiate the BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)
model.cuda()

# Define the optimizer and loss function for training the model
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
loss_fn = torch.nn.CrossEntropyLoss()

# Define the batch size for training and evaluation
batch_size = 4

# Create PyTorch data loaders from the datasets
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=0, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


# Train the model for a specified number of epochs
num_epochs = 2
for epoch in range(num_epochs):
    # Train the model on the training dataset
    model.train()
    for batch in train_loader:
        # Extract the input features and labels from the batch
        input_ids, attention_mask, labels = batch

        # Move the input tensors to the GPU
        input_ids = input_ids.cuda()
        attention_mask = attention_mask.cuda()
        labels = labels.cuda()

        # Forward pass through the model
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        # Backward pass through the model and update the weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate the model on the validation dataset
    model.eval()
    with torch.no_grad():
        total_correct = 0
        total_samples = 0
        total_loss = 0
        for batch in val_loader:
            # Extract the input features and labels from the batch
            input_ids, attention_mask, labels = batch

            # Move the input tensors to the GPU
            input_ids = input_ids.cuda()
            attention_mask = attention_mask.cuda()
            labels = labels.cuda()

            # Forward pass through the model
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            # Compute the accuracy of the model predictions
            _, predicted_labels = torch.max(outputs.logits, dim=1)
            total_correct += torch.sum(predicted_labels == labels)
            total_samples += len(labels)

        accuracy = total_correct / total_samples
        avg_loss = total_loss / len(val_loader)
        print(f'Epoch {epoch + 1} | Val Loss: {avg_loss:.4f} | Val Acc: {accuracy:.4f}')

# Save the trained model
torch.save(model.state_dict(), 'ml_bert_model.pt')


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [ ]:
model.eval()
with torch.no_grad():
    total_correct = 0
    total_samples = 0
    total_loss = 0
    for batch in test_loader:
        # Extract the input features and labels from the batch
        input_ids, attention_mask, labels = batch

        # Move the input tensors to the GPU
        input_ids = input_ids.cuda()
        attention_mask = attention_mask.cuda()
        labels = labels.cuda()
        # Forward pass through the model
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()

        # Compute the accuracy of the model predictions
        _, predicted_labels = torch.max(outputs.logits, dim=1)
        total_correct += torch.sum(predicted_labels == labels)
        total_samples += len(labels)

    accuracy = total_correct / total_samples
    avg = total_loss / len(test_loader)
    print(f'Test Loss: {avg:.4f} | Test Acc: {accuracy:.4f}')

Test Loss: 0.7690 | Test Acc: 0.6990


In [ ]:
import pandas as pd
import gzip
import matplotlib.pyplot as plt
import seaborn as sns

# Load the appliance dataset
with gzip.open('Appliances.json.gz', 'r') as f:
    df = pd.read_json(f, lines=True)
    
df.dropna(subset=['reviewText'], inplace=True)

df['review_length'] = df['reviewText'].apply(lambda x: len(x.split()))


sns.displot(df['review_length'], kde=True, bins=100)
plt.xlabel('Review text length')
plt.ylabel('Count')
plt.title('Distribution of review text length')
plt.xlim(0, 500)
plt.xticks(range(0, 500, 10))
plt.figure(figsize=(50, 10))
plt.show()

In [7]:
# Set maximum number of rows and columns to display
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 6)

# Load the appliance dataset
with gzip.open('Appliances.json.gz', 'r') as f:
    df = pd.read_json(f, lines=True)

df = df[['overall', 'reviewText']]
    
df.dropna(subset=['reviewText'], inplace=True)

df.head(5)



,overall,reviewText
0,5,Not one thing in this book seemed an obvious o...
1,5,I have enjoyed Dr. Alan Gregerman's weekly blo...
2,5,Alan Gregerman believes that innovation comes ...
3,5,"Alan Gregerman is a smart, funny, entertaining..."
4,5,"As I began to read this book, I was again remi..."
